# Mortality risk trajectories (landmark risk scores)

Drives `pipelines.trajectories.generate_mortality_trajectories` as a subprocess. The script scores
every patient's mortality risk at a series of **landmarks** — months 0, 3, 6, ... 60 — refitting the
model at each one on only the notes available up to that landmark, so the resulting per-patient
series is a risk *trajectory* rather than a single baseline score.

Runs **after** [01_run_preprocessing.ipynb](01_run_preprocessing.ipynb) (it reads the note
embeddings and survival cohort directly) and **before**
[06b_generate_figure_data.ipynb](06b_generate_figure_data.ipynb) — `figures.prep.figure4` clusters
these trajectories and derives per-patient risk slopes from them.

### How a landmark is built

At landmark month *M* (day *M*×30), the at-risk set is patients whose **original, unshifted**
`tt_death` is strictly greater than the landmark day. Each landmark is drawn independently from the
full baseline cohort — landmarks are *not* chained through the previous landmark's survivors, which
would compound into a progressively survivor-enriched cohort and make AUCs non-comparable across the
figure's x-axis. Embeddings are re-pooled with `max_note_window=landmark_day` and a time-decay mean
(`decay_param = 0.1`), then a nested held-out Coxnet gives each patient a risk score for that column.

Because the at-risk denominator shrinks with *M*, the script also writes the risk-set size per
landmark. Any comparison of AUC across landmarks has to be read against that denominator.

### Cost and failure behavior — read before starting

This is the **most expensive notebook in the pipeline**: 21 landmarks, each re-pooling the full
embedding array and fitting a nested held-out Coxnet. Expect many hours.

There is **no checkpointing** — unlike the within-vs-pan scripts in
[04c_run_within_vs_pan_models.ipynb](04c_run_within_vs_pan_models.ipynb), this script has no
`RunCheckpoint`, so an interrupted run restarts from month 0 and nothing partial is recoverable.
Start it where it can run to completion.

A landmark that fails is caught, recorded, and skipped, leaving its column all-`NaN`. The script
still writes both CSVs, then **raises at the very end** naming the failed months. So a non-zero exit
here does *not* mean there is no output — it means the output has holes, and the sections below say
which. `figures.prep.figure4` tolerates that: it keeps landmark columns observed in >10% of rows and
requires `MIN_SLOPE_POINTS = 3` observed values per patient within months 0..12.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path


def find_v2_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find v2 root from {start}")


V2_ROOT = find_v2_root()
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

import config
import schemes

MODULE = "pipelines.trajectories.generate_mortality_trajectories"

# Module-level constants in the script, mirrored here for the checks and summaries below.
# Change them in the script, not here.
DECAY_PARAM = 0.1
MONTHS_TO_TEST = [i * 3 for i in range(1, 21)]   # 3..60; column plus_0_months_data is the baseline

TRAJECTORY_PATH = os.path.join(schemes.scheme_results_dir("death_met"), "mortality_trajectories")
TRAJECTORY_CSV = os.path.join(TRAJECTORY_PATH, f"survival_trajectories_w_decay_param_{DECAY_PARAM}.csv")
RISK_SETS_CSV = os.path.join(TRAJECTORY_PATH, f"landmark_risk_sets_w_decay_param_{DECAY_PARAM}.csv")

# figures.prep.figure4 constants, mirrored for the hand-off check.
SLOPE_LANDMARK_MONTHS = 12
MIN_SLOPE_POINTS = 3

# --- Run toggle ---
RUN_TRAJECTORIES = True

# Skip if the trajectory CSV already exists. There is no checkpoint to resume from, so a re-run is
# a full re-run: leave this True unless you actually intend to recompute every landmark.
SKIP_IF_DONE = True

print(f"v2 root:     {V2_ROOT}")
print(f"output dir:  {TRAJECTORY_PATH}")
print(f"landmarks:   0, {', '.join(str(m) for m in MONTHS_TO_TEST[:3])} ... "
      f"{MONTHS_TO_TEST[-1]} months  ({len(MONTHS_TO_TEST) + 1} columns)")

## Preconditions

The script reads the note embeddings, the survival cohort, and the cancer-type table directly — it
does **not** use a prebuilt embedding prediction dataset, so it depends on 01 rather than 03. The
embedding array is the large one; a missing file here is the difference between "the run failed" and
"the run never had its inputs".

In [ ]:
PRECONDITIONS = [
    ("note embeddings meta",  os.path.join(config.NOTES_PATH, "full_clinical_notes_embeddings_metadata.parquet")),
    ("note embeddings array", os.path.join(config.NOTES_PATH, "full_clinical_notes_embeddings_as_array.npy.zst")),
    ("survival cohort",       os.path.join(config.SURV_PATH, "death_met_surv_df.parquet")),
    ("cancer types",          os.path.join(config.FEATURE_PATH, "cancer_type_df.csv.gz")),
]

missing = []
for label, path in PRECONDITIONS:
    ok = os.path.exists(path)
    if not ok:
        missing.append(label)
    size = f"{os.path.getsize(path) / 1e9:>6.2f} GB" if ok else " " * 9
    print(f"[{'ok ' if ok else 'MISSING'}] {label:<22} {size}  {path}")

if missing:
    print(f"\n{len(missing)} input(s) missing: {', '.join(missing)}")
    print("The run will fail. This cell does not raise — inspect and decide.")
else:
    print("\nAll inputs present.")

try:
    import zstandard  # noqa: F401
    print("[ok ] zstandard importable (needed to decompress the embedding array)")
except ImportError:
    print("[MISSING] zstandard not importable — the script cannot read the embedding array. "
          "Run this on the cluster kernel.")

## Run

`python -m pipelines.trajectories.generate_mortality_trajectories` with `cwd` set to `v2/`. Output
streams straight through — this runs for hours behind a per-landmark progress bar, so silence would
be indistinguishable from a hang.

A non-zero exit means at least one landmark failed, **not** that nothing was written. Both CSVs are
written before the script raises; the inventory and results cells below report what actually
landed.

In [ ]:
if not RUN_TRAJECTORIES:
    print("=== disabled (RUN_TRAJECTORIES is False) ===")
elif SKIP_IF_DONE and os.path.exists(TRAJECTORY_CSV):
    mtime = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(TRAJECTORY_CSV)))
    print(f"=== already done (trajectory CSV written {mtime}), skipping ===")
    print(f"    {TRAJECTORY_CSV}")
    print("    Set SKIP_IF_DONE = False to recompute every landmark from scratch.")
else:
    print(f"{'=' * 78}\n=== python -m {MODULE}\n{'=' * 78}", flush=True)
    started = time.time()
    proc = subprocess.run([sys.executable, "-m", MODULE], cwd=str(V2_ROOT))
    elapsed = time.time() - started

    if proc.returncode == 0:
        print(f"\n[ok] all {len(MONTHS_TO_TEST) + 1} landmarks scored in {elapsed / 3600:.2f} h")
    elif os.path.exists(TRAJECTORY_CSV):
        print(f"\n[exit {proc.returncode}] finished in {elapsed / 3600:.2f} h with at least one "
              "failed landmark. The script writes both CSVs before raising, so the output exists "
              "with holes — the coverage table below says which columns are empty.")
    else:
        print(f"\n[exit {proc.returncode}] failed in {elapsed / 3600:.2f} h without writing "
              "anything — the run died before the landmark loop finished (see the traceback "
              "above), not at an individual landmark.")

## Output inventory

What is on disk now. Both files are written in the same step, so one present without the other means
the run died mid-write rather than at a landmark.

In [ ]:
for label, path in [("trajectory scores", TRAJECTORY_CSV), ("landmark risk sets", RISK_SETS_CSV)]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        mtime = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(path)))
        print(f"[ok     ] {label:<20} {size_mb:>8.1f} MB   {mtime}")
        print(f"           {path}")
    else:
        print(f"[missing] {label:<20} {path}")

## Landmark coverage

The one thing worth checking before 06b: how many patients actually got a score at each landmark,
and how that compares to the at-risk denominator the script recorded.

`n_scored` well below `n_at_risk` at a landmark means patients were at risk but dropped for missing
embeddings or covariates. `n_scored = 0` is a failed or skipped landmark — an all-`NaN` column.
Every read is guarded, so this section is safe to run on a partial pipeline.

In [ ]:
import polars as pl

if not os.path.exists(TRAJECTORY_CSV):
    print("Trajectory CSV not written — run has not completed.")
else:
    traj = pl.read_csv(TRAJECTORY_CSV)
    month_cols = [c for c in traj.columns if c != "DFCI_MRN"]

    def _month_num(col: str) -> int:
        return int(col.split("_")[1])

    month_cols = sorted(month_cols, key=_month_num)
    n_rows = traj.height

    scored = {
        c: int(traj.select(pl.col(c).cast(pl.Float64, strict=False).is_finite().sum()).item())
        for c in month_cols
    }

    at_risk = {}
    if os.path.exists(RISK_SETS_CSV):
        rs = pl.read_csv(RISK_SETS_CSV)
        at_risk = dict(zip(rs["months"].to_list(), rs["n_at_risk"].to_list()))
    else:
        print("[note] landmark_risk_sets CSV missing — at-risk denominators unavailable.\n")

    print(f"Cohort: {n_rows:,} patients, {len(month_cols)} landmark columns\n")
    print(f"  {'month':>5}  {'n_scored':>9}  {'n_at_risk':>9}  {'% of cohort':>11}   status")
    empty_months = []
    for c in month_cols:
        m = _month_num(c)
        n = scored[c]
        ar = at_risk.get(m)
        ar_s = f"{ar:,}" if ar is not None else "-"
        pct = 100.0 * n / n_rows if n_rows else 0.0
        if n == 0:
            status = "EMPTY (landmark failed or skipped)"
            empty_months.append(m)
        elif pct <= 10.0:
            status = "below figure4's >10% keep threshold"
        else:
            status = ""
        print(f"  {m:>5}  {n:>9,}  {ar_s:>9}  {pct:>10.1f}%   {status}")

    if empty_months:
        print(f"\n{len(empty_months)} empty landmark(s): "
              f"{', '.join(str(m) for m in empty_months)} months.")

## Hand-off to 06b

`figures.prep.figure4` reads the trajectory CSV, keeps landmark columns observed in more than 10% of
rows, restricts to months 0..`SLOPE_LANDMARK_MONTHS` (12), and requires each patient to have at
least `MIN_SLOPE_POINTS` (3) observed values in that window to get a risk slope. This cell applies
the same three rules so you know what Figure 4 will have to work with before running 06b.

In [ ]:
if not os.path.exists(TRAJECTORY_CSV):
    print("Trajectory CSV not written — not ready for 06b.")
else:
    kept = [c for c in month_cols if scored[c] > 0.1 * n_rows]
    in_window = [c for c in kept if _month_num(c) <= SLOPE_LANDMARK_MONTHS]

    print(f"landmark columns written:                  {len(month_cols)}")
    print(f"  kept by figure4 (>10% observed):         {len(kept)}")
    print(f"  within slope window (months 0..{SLOPE_LANDMARK_MONTHS}):        {len(in_window)}"
          f"   [{', '.join(str(_month_num(c)) for c in in_window)}]")

    if len(in_window) < MIN_SLOPE_POINTS:
        n_slope = 0
        print(f"\n[FAIL] Only {len(in_window)} usable landmark(s) in the slope window, but "
              f"MIN_SLOPE_POINTS = {MIN_SLOPE_POINTS}. No patient can get a risk slope — "
              "figure4 will have nothing to cluster.")
    else:
        observed = (
            traj.select([pl.col(c).cast(pl.Float64, strict=False).is_finite().cast(pl.Int32)
                         for c in in_window])
            .sum_horizontal()
        )
        n_slope = int((observed >= MIN_SLOPE_POINTS).sum())
        print(f"\npatients with >= {MIN_SLOPE_POINTS} observed landmarks in the window: "
              f"{n_slope:,} / {n_rows:,}  ({100.0 * n_slope / n_rows:.1f}%)")

    print(f"\nfigure data dir: {config.FIGURE_DATA_DIR}")
    for csv in ["fig4_trajectories_heatmap.csv", "fig4_km_data.csv",
                "fig4_cluster_severity.csv", "fig4_group_trajectories.csv",
                "fig4_slope_by_stage.csv", "fig4_silhouette.csv"]:
        fig_path = os.path.join(config.FIGURE_DATA_DIR, csv)
        if os.path.exists(fig_path):
            mtime = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(fig_path)))
            print(f"  [existing] {csv:<30} last written {mtime}")
        else:
            print(f"  [none    ] {csv:<30} (06b has not written it yet)")

    print("\n" + ("Ready for 06b_generate_figure_data.ipynb."
                   if n_slope else
                   "Not ready — re-run the failed landmarks before 06b."))